# Latency benchmark — corrected

Per-query time for retrieval alone vs retrieval + reranking, so the paper can state a real number
rather than calling the mitigation "cheap" without evidence. Reports **median and p95**, not just
mean — p95 is the worst case users notice.

One-time costs (index building, model loading) are measured **separately** from per-query cost.

## Defects fixed

**Paths.** The original opened `corpus_v2.json` / `qa_pairs_wiki.json` from the working directory;
in this repo they are `data/corpus.json` and `data/qa_pairs_wiki.json`. Outputs went to the working
directory rather than `results/`, and the last cell was a bare `from google.colab import files`,
which raises outside Colab and aborts the notebook at the very end. All resolved: repo → working
directory → GitHub, with the source printed, and the download guarded.

**Deprecated `torch_dtype`.** `automodel_args={"torch_dtype": ...}` is deprecated in current
transformers and absent in old ones. Replaced with a post-load cast that works on every version.

**Warmup called `timed_retrieve` twice per query.** Harmless but wasteful, and it obscured that the
reranker warmup depended on the second call's output. Now one call, result reused.

**The reranker was loaded in float32.** On a GPU that is roughly half the speed of float16, so the
measured reranking overhead was pessimistic relative to how anyone would actually deploy it. The
dtype is now a config field, defaulting to float16 on CUDA and float32 elsewhere, and the summary
states which was used — a latency number without its dtype is not reproducible.

## Checked and *not* a bug

`ce.predict()` and `bi.encode()` both return numpy arrays, and that conversion forces a CUDA
synchronise. So `perf_counter` around them does measure completed GPU work — the timings are valid
as written. (This is *not* true on XLA; see the TPU variant.)

### Install

In [ ]:
import importlib.util, subprocess, sys

need = [p for p, m in [("rank_bm25", "rank_bm25"), ("sentence-transformers", "sentence_transformers"), ("transformers", "transformers")] if importlib.util.find_spec(m) is None]
if need:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=False)

print("deps ready")

### Paths

In [ ]:
import os, json, re, gc, time, random, pickle, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd

RAW_BASE = "https://raw.githubusercontent.com/Rania-khaoudane/MSA/main/data"

def _repo_root():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "data" / "qa_pairs_wiki.json").exists():
            return base
    return None

ROOT = _repo_root()

def find_file(*names, subdirs=("data", "results")):
    """Locate a file: repo subdirs first, then the working directory."""
    for n in names:
        if ROOT:
            for sd in subdirs:
                p = ROOT / sd / n
                if p.exists():
                    return p
        if Path(n).exists():
            return Path(n)
    return None

def resolve(*names):
    """Local -> GitHub. Returns (json, description)."""
    p = find_file(*names, subdirs=("data",))
    if p:
        return json.loads(p.read_text(encoding="utf-8")), f"local: {p}"
    for n in names:
        try:
            url = f"{RAW_BASE}/{n}"
            with urllib.request.urlopen(url) as r:
                return json.loads(r.read().decode("utf-8")), f"github: {url}"
        except Exception:
            continue
    raise FileNotFoundError(f"none of {names} found locally or on GitHub")

OUT_DIR = (ROOT / "results") if ROOT else Path(".")
OUT_DIR.mkdir(exist_ok=True)
def out(name):
    return str(OUT_DIR / name)

print("repo root :", ROOT or "(not in the repo)")
print("output dir:", OUT_DIR.resolve())

import torch

### Config

In [ ]:
CONFIG = {
    "base_encoder": "intfloat/multilingual-e5-base",
    "reranker": "BAAI/bge-reranker-v2-m3",
    "alpha": 0.8,
    "retrieve_k": 20,   # candidates passed to the reranker -- same as solution_6
    "n_queries": 100,   # timed queries, after warmup
    "n_warmup": 5,
    "max_length": 512,
    "batch_size": 16,
    "seed": 42,
}
CONFIG

### Load data (index-build time is NOT per-query cost)

In [ ]:
corpus, src_c = resolve("corpus_v2.json", "corpus.json")
wiki_qa, src_q = resolve("qa_pairs_wiki.json")
print("corpus from", src_c); print("qa     from", src_q)

corpus_ids = [c["chunk_id"] for c in corpus]
corpus_texts = [c["text"] for c in corpus]
corpus_map = dict(zip(corpus_ids, corpus_texts))
known = set(corpus_ids)
qa = [q for q in wiki_qa if q["source_chunk_id"] in known]
random.Random(CONFIG["seed"]).shuffle(qa)

need = CONFIG["n_queries"] + CONFIG["n_warmup"]
if len(qa) < need:
    raise ValueError(f"need {need} queries (n_queries + n_warmup) but only {len(qa)} available")
timed_qa = qa[: CONFIG["n_queries"]]
warmup_qa = qa[CONFIG["n_queries"]: need]
print(f"Corpus: {len(corpus)} passages | timing {len(timed_qa)} queries "
      f"(+{len(warmup_qa)} warmup, excluded from the measurement)")

### BM25 (one-time build cost)

In [ ]:
from rank_bm25 import BM25Okapi

DIAC = re.compile(r"[\u0610-\u061A\u064B-\u065F\u06D6-\u06DC\u06DF-\u06E8\u06EA-\u06ED\u0670]")

def normalize_arabic(t):
    t = DIAC.sub("", t)
    t = re.sub(r"[\u0625\u0623\u0622\u0627]", "\u0627", t)
    t = re.sub(r"\u0649", "\u064A", t); t = re.sub(r"\u0629", "\u0647", t)
    t = re.sub(r"\u0624", "\u0648", t); t = re.sub(r"\u0626", "\u064A", t)
    t = re.sub(r"\u0640+", "", t); t = re.sub(r"[^\w\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

def tokenize(t):
    return normalize_arabic(t).split()

t0 = time.perf_counter()
bm25 = BM25Okapi([tokenize(t) for t in corpus_texts])
bm25_build_s = time.perf_counter() - t0
print(f"BM25 index built in {bm25_build_s:.2f}s (one-time, not per-query).")

def minmax(a):
    lo, hi = a.min(), a.max()
    return (a - lo) / (hi - lo) if hi > lo else np.zeros_like(a)

### Build the dense index and load the reranker (one-time costs)

In [ ]:
from sentence_transformers import SentenceTransformer, CrossEncoder

bi = SentenceTransformer(CONFIG["base_encoder"])
t0 = time.perf_counter()
corpus_emb = np.asarray(
    bi.encode([f"passage: {t}" for t in corpus_texts],
              normalize_embeddings=True, batch_size=32, show_progress_bar=True), "float32")
index_build_s = time.perf_counter() - t0
print(f"Dense index built in {index_build_s:.2f}s for {len(corpus)} passages (one-time).")

device_name = "cuda" if torch.cuda.is_available() else "cpu"
# float32 on a GPU is roughly half the speed of float16, which made the original
# measurement pessimistic relative to any real deployment. State the dtype used:
# a latency figure without it is not reproducible.
RERANK_DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

t0 = time.perf_counter()
ce = CrossEncoder(CONFIG["reranker"], max_length=CONFIG["max_length"], trust_remote_code=True)
try:
    ce.model = ce.model.to(dtype=RERANK_DTYPE)
except Exception as e:
    print(f"  dtype cast skipped ({type(e).__name__}); using the loaded default.")
    RERANK_DTYPE = next(ce.model.parameters()).dtype
model_load_s = time.perf_counter() - t0
print(f"Reranker loaded in {model_load_s:.2f}s: {CONFIG['reranker']} ({RERANK_DTYPE})")
print(f"Device: {device_name}")

### Timed stages

In [ ]:
def timed_retrieve(query, k):
    """Hybrid retrieval for one query -> (candidate_ids, elapsed_seconds).

    bi.encode() returns numpy, which forces a CUDA synchronise, so this timer
    does measure completed GPU work.
    """
    t0 = time.perf_counter()
    q = bi.encode([f"query: {query}"], normalize_embeddings=True)[0]
    s = (CONFIG["alpha"] * minmax(corpus_emb @ q)
         + (1 - CONFIG["alpha"]) * minmax(np.asarray(bm25.get_scores(tokenize(query)))))
    idx = np.argsort(-s)[:k]
    return [corpus_ids[i] for i in idx], time.perf_counter() - t0

def timed_rerank(query, candidate_ids):
    """Cross-encoder rerank -> (reordered_ids, elapsed_seconds)."""
    t0 = time.perf_counter()
    pairs = [(query, corpus_map[c]) for c in candidate_ids]
    scores = np.asarray(ce.predict(pairs, batch_size=CONFIG["batch_size"], show_progress_bar=False))
    if scores.ndim > 1:
        scores = scores[:, -1]
    order = np.argsort(-scores)
    return [candidate_ids[i] for i in order], time.perf_counter() - t0

### Warmup (excluded from the measurement)

In [ ]:
print("Warming up...")
t0 = time.perf_counter()
first_call_s = None
for i, q in enumerate(warmup_qa):
    cands, t_r = timed_retrieve(q["darija_query"], CONFIG["retrieve_k"])   # one call, result reused
    _, t_rr = timed_rerank(q["darija_query"], cands)
    if i == 0:
        first_call_s = t_r + t_rr
warmup_s = time.perf_counter() - t0
print(f"Warmup done in {warmup_s:.2f}s; first call alone took {first_call_s*1000:.0f} ms"
      + ("  <- includes XLA compilation" if "BACKEND" in dir() and BACKEND == "tpu" else ""))

### The timed run

In [ ]:
rows = []
for q in timed_qa:
    cands, t_retrieve = timed_retrieve(q["darija_query"], CONFIG["retrieve_k"])
    _, t_rerank = timed_rerank(q["darija_query"], cands)
    rows.append({"qid": q["id"], "retrieve_ms": t_retrieve * 1000,
                 "rerank_ms": t_rerank * 1000,
                 "total_ms": (t_retrieve + t_rerank) * 1000})

lat = pd.DataFrame(rows)
lat.to_csv(out("latency_measurements.csv"), index=False)
print(f"Timed {len(lat)} queries.")

### Summary statistics (the numbers for the paper)

In [ ]:
def stats(s):
    return {"mean": s.mean(), "median": s.median(), "p95": s.quantile(0.95),
            "min": s.min(), "max": s.max()}

DEV = device_name
print("=" * 72)
print(f"LATENCY SUMMARY (n={len(lat)} queries, device={DEV})")
print("=" * 72)
print(f"  reranker dtype    {RERANK_DTYPE}")
for col, label in [("retrieve_ms", "Retrieval only (BM25 + dense)"),
                   ("rerank_ms", f"Reranking only ({CONFIG['reranker'].split('/')[-1]})"),
                   ("total_ms", "Total (retrieval + reranking)")]:
    s = stats(lat[col])
    print(f"\n{label}:")
    print(f"  mean {s['mean']:.1f} ms | median {s['median']:.1f} ms | "
          f"p95 {s['p95']:.1f} ms | range [{s['min']:.1f}, {s['max']:.1f}] ms")

overhead = (lat.rerank_ms.mean() / lat.retrieve_ms.mean()) * 100
print(f"\nReranking adds {lat.rerank_ms.mean():.1f} ms on top of "
      f"{lat.retrieve_ms.mean():.1f} ms retrieval ({overhead:.0f}% relative overhead).")

print(f"""
One-time costs (not per-query):
  BM25 index build     {bm25_build_s:.2f} s  for {len(corpus)} passages
  Dense index build    {index_build_s:.2f} s  for {len(corpus)} passages
  Reranker load        {model_load_s:.2f} s
  Warmup               {warmup_s:.2f} s (first call {first_call_s*1000:.0f} ms)

Suggested paper sentence:
  "On a {str(DEV).upper()}, hybrid retrieval over {len(corpus)} passages took a median of
  {stats(lat['retrieve_ms'])['median']:.0f} ms per query; adding {CONFIG['reranker'].split('/')[-1]}
  reranking of the top-{CONFIG['retrieve_k']} candidates added a median of
  {stats(lat['rerank_ms'])['median']:.0f} ms ({overhead:.0f}% relative overhead), for a total of
  {stats(lat['total_ms'])['median']:.0f} ms per query (p95: {stats(lat['total_ms'])['p95']:.0f} ms)."
""")

print(f"\nAll outputs written under: {OUT_DIR.resolve()}")
for f in ["latency_measurements.csv"]:
    print("  ", f)

# The original ended with a bare `from google.colab import files`, which raises
# outside Colab and aborted the notebook on the final cell.
try:
    from google.colab import files as colab_files
    for f in ["latency_measurements.csv"]:
        colab_files.download(out(f))
except ImportError:
    print("(Not in Colab - files are on disk at the path above.)")